
# Running Total of Sales (E-commerce Company)

## Business Problem

Ane-commerce company wants to track how much revenue each customer has generated over time.

The business asks:

> "For every new purchase, show the customer's cumulative spend till that date."

This helps in:

- find Premium Membership Eligibility
- Assign Loyalty Tiers


- when(col("cumulative_spend") >= 75000, "Platinum")
-          .when(col("cumulative_spend") >= 50000, "Gold")
-          .when(col("cumulative_spend") >= 25000, "Silver")
-          .otherwise("Bronze")



## Sample Data

|transaction_id|customer_id|customer_name|order_id|product_name|sales_amount|purchase_date|
|---|---|---|---|---|---|---|
|1001|C101|Amit Sharma|ORD-001|Laptop|65000|2026-01-05|
|1003|C101|Amit Sharma|ORD-003|Headphones|3500|2026-01-10|
|1006|C101|Amit Sharma|ORD-006|Wireless Mouse|1500|2026-01-18|
|1011|C101|Amit Sharma|ORD-011|Office Chair|12500|2026-02-02|
|1015|C101|Amit Sharma|ORD-015|Web Camera|4200|2026-02-15|
|1002|C102|Neha Verma|ORD-002|Mobile Phone|28000|2026-01-06|
|1005|C102|Neha Verma|ORD-005|Laptop Bag|2200|2026-01-15|
|1009|C102|Neha Verma|ORD-009|Monitor|17500|2026-01-25|
|1013|C102|Neha Verma|ORD-013|USB Hub|1800|2026-02-08|
|1004|C103|Rahul Singh|ORD-004|Smart Watch|8500|2026-01-12|


## Expected Result for Customer C101:

|transaction_id|customer_id|customer_name|order_id|product_name|sales_amount|purchase_date|cumulative_spend|
|---|---|---|---|---|---|---|---|
|1001|C101|Amit Sharma|ORD-001|Laptop|65000|2026-01-05|65000|
|1003|C101|Amit Sharma|ORD-003|Headphones|3500|2026-01-10|68500|
|1006|C101|Amit Sharma|ORD-006|Wireless Mouse|1500|2026-01-18|70000|
|1011|C101|Amit Sharma|ORD-011|Office Chair|12500|2026-02-02|82500|
|1015|C101|Amit Sharma|ORD-015|Web Camera|4200|2026-02-15|86700|

## Window Frame: `rowsBetween()`

`rowsBetween()` is a **WindowSpec object method** that defines the frame boundary — *which rows* are included in the aggregation for each row.

### Syntax

```python
Window.rowsBetween(start, end)
```

### Example used in this notebook

```python
.rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)
```

This means: **start summing from the very first row of the partition, all the way down to the current row** — a running total.

### Common boundary values

| Value | Meaning |
|---|---|
| `Window.unboundedPreceding` | First row of the partition |
| `Window.currentRow` (or `0`) | The current row |
| `Window.unboundedFollowing` | Last row of the partition |
| `-1` | 1 row before the current row |
| `1` | 1 row after the current row |

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from datetime import datetime

sales_data = [
    (1001, "C101", "Amit Sharma",  "ORD-001", "Laptop",         65000.00, datetime.strptime("2026-01-05", "%Y-%m-%d").date()),
    (1002, "C102", "Neha Verma",   "ORD-002", "Mobile Phone",   28000.00, datetime.strptime("2026-01-06", "%Y-%m-%d").date()),
    (1003, "C101", "Amit Sharma",  "ORD-003", "Headphones",      3500.00, datetime.strptime("2026-01-10", "%Y-%m-%d").date()),
    (1004, "C103", "Rahul Singh",  "ORD-004", "Smart Watch",     8500.00, datetime.strptime("2026-01-12", "%Y-%m-%d").date()),
    (1005, "C102", "Neha Verma",   "ORD-005", "Laptop Bag",      2200.00, datetime.strptime("2026-01-15", "%Y-%m-%d").date()),
    (1006, "C101", "Amit Sharma",  "ORD-006", "Wireless Mouse",  1500.00, datetime.strptime("2026-01-18", "%Y-%m-%d").date()),
    (1007, "C104", "Priya Patel",  "ORD-007", "Tablet",         32000.00, datetime.strptime("2026-01-20", "%Y-%m-%d").date()),
    (1008, "C103", "Rahul Singh",  "ORD-008", "Keyboard",        2800.00, datetime.strptime("2026-01-22", "%Y-%m-%d").date()),
    (1009, "C102", "Neha Verma",   "ORD-009", "Monitor",        17500.00, datetime.strptime("2026-01-25", "%Y-%m-%d").date()),
    (1010, "C104", "Priya Patel",  "ORD-010", "Tablet Cover",    1200.00, datetime.strptime("2026-01-27", "%Y-%m-%d").date()),
    (1011, "C101", "Amit Sharma",  "ORD-011", "Office Chair",   12500.00, datetime.strptime("2026-02-02", "%Y-%m-%d").date()),
    (1012, "C103", "Rahul Singh",  "ORD-012", "Mobile Phone",   24000.00, datetime.strptime("2026-02-05", "%Y-%m-%d").date()),
    (1013, "C102", "Neha Verma",   "ORD-013", "USB Hub",         1800.00, datetime.strptime("2026-02-08", "%Y-%m-%d").date()),
    (1014, "C104", "Priya Patel",  "ORD-014", "Bluetooth Speaker", 4500.00, datetime.strptime("2026-02-10", "%Y-%m-%d").date()),
    (1015, "C101", "Amit Sharma",  "ORD-015", "Web Camera",      4200.00, datetime.strptime("2026-02-15", "%Y-%m-%d").date())
]

sales_schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("sales_amount", DoubleType(), False),
    StructField("purchase_date", DateType(), False)
])

sales_df = spark.createDataFrame(sales_data, sales_schema)

display(sales_df.orderBy("customer_id", "purchase_date"))

In [0]:
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

running_total_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("purchase_date", "transaction_id") # one cust can multiple order on single date so transaction_id
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)


In [0]:
customer_sales_running_total_df = (
    sales_df
    .withColumn(
        "cumulative_spend",
        sum("sales_amount").over(running_total_window)
    )
    .orderBy("customer_id", "purchase_date", "transaction_id")
)

display(customer_sales_running_total_df)

In [0]:
from pyspark.sql.functions import when, col
loyalty_df = (
    customer_sales_running_total_df
    .withColumn(
        "loyalty_tier",
        when(col("cumulative_spend") >= 75000, "Platinum")
         .when(col("cumulative_spend") >= 50000, "Gold")
         .when(col("cumulative_spend") >= 25000, "Silver")
         .otherwise("Bronze")
    )
)

display(loyalty_df)

### Find running total only for the last 3 transactions.

In [0]:
running_total_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("purchase_date", "transaction_id") 
    .rowsBetween(-2, 0)
)

last_3_txn_df = sales_df.withColumn(
    "last_3_txn_total",
    sum("sales_amount").over(running_total_window)
)

last_3_txn_df.display()